### **Setup: Imports und Initialisierung**

In [ ]:
import sys
import os
import glob
sys.path.append(os.path.abspath('..'))

from quinex_application.quinex_processing.scopus_data import ProcessScopusData

In [ ]:
output_path = "../data/paper/paper_dataset.parquet"
csvs_paths = glob.glob("../data/paper/wos_query4_closedaccess_new/by_year/wos_*.xls")
query_name = 'Query: TS=("energy system*" AND (simul* OR model* OR plan OR plans OR planning OR optimi* OR analy* OR assess* OR evaluat*))'
query_date = "12.11.2025"
quelle_name = "Source: WOS"

scopus = ProcessScopusData(csvs_paths, output_path, query_name, query_date, quelle_name) 

## **Load and Clean CSV Files**

### **Combine the CSV Files**

In [ ]:
df = scopus.combine_csvs()
print(len(df))

### **Clean duplicate DOIs**

In [ ]:
duplicate_doi_values = df["DOI"].value_counts()
duplicate_doi_values = duplicate_doi_values[duplicate_doi_values > 1]
print(f"Anzahl verschiedener mehrfacher DOIs: {len(duplicate_doi_values)}")
print(f"Anzahl an Zeilen mit mehrfacher DOI: {duplicate_doi_values.sum()}")

In [ ]:
df_cleaned = scopus.remove_duplicates_doi()
print(len(df_cleaned))

### **Add new Papers to Parquet**

In [ ]:
parquet = scopus.update_parquet_with_new_data()

In [ ]:
parquet[
    (
        (parquet['Query: AB=("energy system*" AND (simul* OR model* OR plan OR plans OR planning OR optimi* OR analy* OR assess* OR evaluat*))_24.10.2025'] == True)
        & 
        (parquet['Query: TS=("energy system*" AND (simul* OR model* OR plan OR plans OR planning OR optimi* OR analy* OR assess* OR evaluat*))_12.11.2025'] == False)
    )
]

In [ ]:
print("-------------------Overall Papers-------------------------------")
print(len(parquet[
    (
        (parquet['Query: AB=("energy system*" AND (simul* OR model* OR plan OR plans OR planning OR optimi* OR analy* OR assess* OR evaluat*))_10.06.2025'] == True)
        | 
        (parquet['Query: TS=("energy system*" AND (simul* OR model* OR plan OR plans OR planning OR optimi* OR analy* OR assess* OR evaluat*))_12.11.2025'] == True)
        | 
        (parquet['Query: TITLE-ABS-KEY ( ( "energy system*" ) AND ( simul* OR model* OR "plan" OR "plans" OR "planning" OR optimi* OR analy* OR assess* OR evaluat* ) ) AND ( LIMIT-TO ( OA , "all" ) ) AND ( LIMIT-TO ( LANGUAGE , "English" ) )_14.05.2025'] == True)
        |
        (parquet['Query: TITLE-ABS-KEY ( ( "energy system*" ) AND ( simul* OR model* OR "plan" OR "plans" OR "planning" OR optimi* OR analy* OR assess* OR evaluat* ) ) AND ( LIMIT-TO ( LANGUAGE , "English" ) )_22.10.2025'] == True)
    )
    & 
    (parquet['Year'] >= 2010)
]))

### **Save cleaned and added data**

In [25]:
scopus.save_df()

## **Select specific Query**

In [1]:
import pandas as pd
df = pd.read_parquet("../data/paper/paper_dataset.parquet")

In [11]:
df.columns

Index(['Year', 'Title', 'Source title', 'Publisher', 'Cited by', 'DOI', 'Link',
       'Abstract', 'Open Access', 'Source: Scopus', 'Source: WOS',
       'Query: TITLE-ABS-KEY ( ( "energy system*" ) AND ( simul* OR model* OR "plan" OR "plans" OR "planning" OR optimi* OR analy* OR assess* OR evaluat* ) ) AND ( LIMIT-TO ( OA , "all" ) ) AND ( LIMIT-TO ( LANGUAGE , "English" ) )_14.05.2025',
       'Query: AB=("energy system*" AND (simul* OR model* OR plan OR plans OR planning OR optimi* OR analy* OR assess* OR evaluat*))_10.06.2025',
       'Query: TITLE-ABS-KEY ( ( "energy system*" ) AND ( simul* OR model* OR "plan" OR "plans" OR "planning" OR optimi* OR analy* OR assess* OR evaluat* ) ) AND ( LIMIT-TO ( LANGUAGE , "English" ) )_22.10.2025',
       'Query: TS=("energy system*" AND (simul* OR model* OR plan OR plans OR planning OR optimi* OR analy* OR assess* OR evaluat*))_12.11.2025',
       'Quinex', 'Quinex_ID', 'Quinex_Version', 'Lizenz', 'OEO_Version',
       'Download_Status'],
   

In [10]:
print("-------------------QUINEX Performed-------------------------------")
print(len(df[df["Quinex"] == True]))
print("-------------------QUINEX Tried-----------------------------------")
print(len(df[(df["Download_Status"] == "Succeeded")]) +  len(df[(df["Download_Status"] == "Abstract Succeeded")]) + len(df[(df["Download_Status"] == "Failed")]))
print("-------------------Overall Papers-------------------------------")
print(len(df[
    (
        (df['Query: AB=("energy system*" AND (simul* OR model* OR plan OR plans OR planning OR optimi* OR analy* OR assess* OR evaluat*))_10.06.2025'] == True)
        | 
        (df['Query: TS=("energy system*" AND (simul* OR model* OR plan OR plans OR planning OR optimi* OR analy* OR assess* OR evaluat*))_12.11.2025'] == True)
        | 
        (df['Query: TITLE-ABS-KEY ( ( "energy system*" ) AND ( simul* OR model* OR "plan" OR "plans" OR "planning" OR optimi* OR analy* OR assess* OR evaluat* ) ) AND ( LIMIT-TO ( OA , "all" ) ) AND ( LIMIT-TO ( LANGUAGE , "English" ) )_14.05.2025'] == True)
        |
        (df['Query: TITLE-ABS-KEY ( ( "energy system*" ) AND ( simul* OR model* OR "plan" OR "plans" OR "planning" OR optimi* OR analy* OR assess* OR evaluat* ) ) AND ( LIMIT-TO ( LANGUAGE , "English" ) )_22.10.2025'] == True)
    )
    & 
    (df['Year'] >= 2010)
]))

-------------------QUINEX Performed-------------------------------
57523
-------------------QUINEX Tried-----------------------------------
76306
-------------------Overall Papers-------------------------------
76306


In [4]:
print(len(df[(df["Download_Status"] == "Abstract Succeeded")]))
print(len(df[(df["Download_Status"] == "Succeeded")]))
print(len(df[(df["Download_Status"] == "Failed")]))

41229
30343
427
